In [0]:
%pip install folium --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import folium
import pandas as pd
import numpy as np
import base64

# Load data
df = spark.table("default.ghana_gap_analysis").toPandas()
df['is_medical_desert'] = df['is_medical_desert'].apply(lambda x: str(x) == 'True')
df['has_anomaly'] = df['has_anomaly'].apply(lambda x: str(x) == 'True')

city_coords = {
    "Accra": (5.6037, -0.1870), "Kumasi": (6.6885, -1.6244),
    "Tamale": (9.4008, -0.8393), "Takoradi": (4.8845, -1.7554),
    "Cape Coast": (5.1054, -1.2466), "Koforidua": (6.0940, -0.2574),
    "Sunyani": (7.3349, -2.3123), "Ho": (6.6011, 0.4712),
    "Tema": (5.6698, -0.0166), "Obuasi": (6.2000, -1.6667),
    "Techiman": (7.5892, -1.9344), "Tarkwa": (5.3000, -1.9833),
    "Akwatia": (6.0333, -0.8000), "Hohoe": (7.1500, 0.4667),
    "Nsawam": (5.8000, -0.3500), "Dodowa": (5.8833, 0.0167),
    "Kasoa": (5.5333, -0.4167), "Madina": (5.6833, -0.1667),
    "Ashaiman": (5.7000, -0.0333), "Winneba": (5.3500, -0.6333),
    "Bolgatanga": (10.7856, -0.8514), "Wa": (10.0601, -2.5099),
    "Duayaw Nkwanta": (7.1667, -2.1000), "Nkwanta": (8.0000, 0.5000),
    "Walewale": (10.3500, -0.8000), "Berekum": (7.4500, -2.5833),
}

def get_coords(row):
    city = str(row.get('address_city', ''))
    for key, coords in city_coords.items():
        if key.lower() in city.lower():
            return coords
    return None

df['coords'] = df.apply(get_coords, axis=1)
df_mapped = df[df['coords'].notna()].copy()

print("Total mapped:", len(df_mapped))
print("Deserts:", df_mapped['is_medical_desert'].sum())
print("Anomalies:", df_mapped['has_anomaly'].sum())
print("Adequate:", (~df_mapped['is_medical_desert'] & ~df_mapped['has_anomaly']).sum())

Total mapped: 635
Deserts: 413
Anomalies: 67
Adequate: 175


In [0]:
m = folium.Map(location=[7.9465, -1.0232], zoom_start=7, tiles="CartoDB dark_matter")

for _, row in df_mapped.iterrows():
    lat, lon = row['coords']
    is_desert = bool(row['is_medical_desert'])
    has_anomaly = bool(row['has_anomaly'])

    lat = lat + np.random.uniform(-0.05, 0.05)
    lon = lon + np.random.uniform(-0.05, 0.05)

    if has_anomaly:
        hex_color = "#FF8C00"
    elif is_desert:
        hex_color = "#CC0000"
    else:
        hex_color = "#006400"

    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color="#000000",
        weight=1,
        fill=True,
        fill_color=hex_color,
        fill_opacity=1.0,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>"
            f"City: {row['address_city']}<br>"
            f"Desert Score: {row['desert_score']}<br>"
            f"Anomaly: {has_anomaly}",
            max_width=200
        ),
        tooltip=row['name']
    ).add_to(m)

legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:#1a1a2e; color:white; padding:15px; border-radius:10px;
     border:2px solid #444; font-family:Arial; font-size:13px">
    <b>Ghana Medical Facility Map</b><br><br>
    <span style="color:#CC0000; font-size:18px">●</span> Medical Desert<br>
    <span style="color:#FF8C00; font-size:18px">●</span> Suspicious Facility<br>
    <span style="color:#006400; font-size:18px">●</span> Adequate Facility<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
m.save("/tmp/ghana_final_map.html")

with open("/tmp/ghana_final_map.html", "rb") as f:
    encoded = base64.b64encode(f.read()).decode()

displayHTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600px"></iframe>')